# 02-流式输出 - Kimi API

本文档演示如何使用 Kimi API 的流式输出功能。

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# 加载 .env 文件
load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.cn/v1")

client = OpenAI(api_key=api_key, base_url=base_url)
print("✅ Kimi 客户端初始化成功")

✅ Kimi 客户端初始化成功


## 基础流式输出

In [2]:
# 基础流式调用
response = client.chat.completions.create(
    model="kimi-k2-turbo-preview",
    messages=[{"role": "user", "content": "Hello!"}],
    stream=True,
)

print("流式回复: ", end="")
collected_content = []

for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        collected_content.append(content)
        print(content, end="", flush=True)

print("\n")
print(f"完整内容: {''.join(collected_content)}")

流式回复: Hello! How can I help you today?

完整内容: Hello! How can I help you today?


## 带 Token 统计的流式输出

In [3]:
def stream_chat_with_stats(prompt: str):
    """带统计信息的流式对话"""
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        stream_options={"include_usage": True},
    )
    
    collected_content = []
    usage = None
    
    for chunk in response:
        delta = chunk.choices[0].delta
        
        if delta.content:
            collected_content.append(delta.content)
            print(delta.content, end="", flush=True)
        
        # 获取 usage 信息
        if chunk.usage:
            usage = chunk.usage
    
    print("\n")
    
    if usage:
        print("Token 统计:")
        print(f"  Prompt tokens: {usage.prompt_tokens}")
        print(f"  Completion tokens: {usage.completion_tokens}")
        print(f"  Total tokens: {usage.total_tokens}")
    
    return "".join(collected_content)

result = stream_chat_with_stats("简要解释什么是人工智能")

人工智能（AI）是让机器模拟人类智能的技术，使计算机能“看、听、想、动”，完成识别图像、理解语言、推理决策等任务。

IndexError: list index out of range

## 打字机效果

In [ ]:
import time

def typewriter_effect(prompt: str, delay: float = 0.02):
    """打字机效果输出"""
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)
            time.sleep(delay)
    print()

# 示例（设置较短的延迟便于演示）
typewriter_effect("用三句话讲一个短故事", delay=0.01)

Once upon a time...


## 流式输出 + 思考模式

In [ ]:
# 流式思考模式
response = client.chat.completions.create(
    model="kimi-k2.5",
    messages=[{"role": "user", "content": "解方程 2x + 5 = 13"}],
    thinking={"type": "enabled"},
    stream=True,
)

print("思考过程: ")
is_thinking = True

for chunk in response:
    delta = chunk.choices[0].delta
    
    # 思考内容
    if hasattr(delta, 'reasoning_content') and delta.reasoning_content:
        print(delta.reasoning_content, end="", flush=True)
    
    # 切换到正式回答
    if delta.content and is_thinking:
        is_thinking = False
        print("\n\n最终答案: ")
    
    if delta.content:
        print(delta.content, end="", flush=True)

print()

思考过程: 
让我思考...

最终答案: 
答案内容...


## 不同模型的流式输出对比

In [ ]:
prompt = "解释什么是机器学习"

models = ["kimi-k2-turbo-preview", "kimi-k2-thinking"]

for model in models:
    print(f"\n=== {model} ===")
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            stream=True,
        )
        
        for chunk in response:
            content = chunk.choices[0].delta.content
            if content:
                print(content, end="", flush=True)
        print()
    except Exception as e:
        print(f"错误: {e}")

=== kimi-k2-turbo-preview ===
回复内容...

=== kimi-k2-thinking ===
回复内容...
